In [1]:
# from 03-search-evaluation.ipynb
from ingest import load_faq_data, build_index
import pandas as pd
from pprint import pprint
from tqdm.auto import tqdm

# read the ground truth data created
df_ground_truth = pd.read_csv(filepath_or_buffer='data/ground_truth_new.csv')
ground_truth = df_ground_truth.to_dict(orient='records')

# load the faq documents and index them
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc['course'] == 'llm-zoomcamp':
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

# define the function to perform text search using the index created
def text_search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    return index.search(query=query, num_results=5, boost_dict=boost_dict)

# function to create the relevance list for one ground truth question; we pass the search function as an argument
def compute_relevance(q, search_function):
    doc_id = q['document']
    results = search_function(query=q['question'])

    relevance = []
    for d in results:
        relevance.append(int(d['id'] == doc_id))

    return relevance

# function to create the relevance lists for more than one ground truth questions; we pass the search function as an argument
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

relevance_total = compute_relevance_total(ground_truth, text_search)

def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + (1 / (rank + 1))
                break
                 
    return total_score / len(relevance)

# function that computes the relevance lists using a search function, and the search metrics
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    
    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

  0%|          | 0/565 [00:00<?, ?it/s]

In [ ]:
# we boosted question to 3.0 in text_search(); the idea was that a query should match the faq question, and that kind of match should 
# count for more than matching the answer text; it sounds reasonable, but it's a guess; we can check it against data instead of 
# trusting it - this is the main benefit of offline evaluation - we change one parameter, run the same questions again, and see whether 
# the metric moves - the dataset stays fixed, so the comparison is fair

# search function with configurable question boost
def search_boost(query, question_boost):
    boost_dict = {'question': question_boost, 'section': 0.5}
    return index.search(query=query, num_results=5, boost_dict=boost_dict)

for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    # lambda query, boost=boost - a small inline function that takes two parameters: query (required) and boost (optional, with a default value)
    # the boost on the right side of 'boost=boost' refers to whatever boost variable exists in the enclosing scope at the moment the 
    # lambda is created - boost on the left-hand side is the lambda's own parameter name
    result = evaluate(ground_truth=ground_truth, search_function=lambda query, boost=boost: search_boost(query, boost))
    print(f'boost={boost}: {result}')
    # increasing the question boost makes the metrics worse, not better - the best value here is 1.0, no boost at all -
    # that's already the opposite of what the intuition predicted

  0%|          | 0/565 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.9008849557522124, 'mrr': 0.7975221238938051}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.904424778761062, 'mrr': 0.8074041297935098}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.863716814159292, 'mrr': 0.7317994100294983}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.856637168141593, 'mrr': 0.7094100294985247}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.8424778761061947, 'mrr': 0.6888790560471973}


In [ ]:
# a search function with all three boosts - question, answer and section (for demo - we use lambda function in the main grid search)
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {'question': question_boost, 'answer': answer_boost, 'section': section_boost}
    return index.search(query=query, num_results=5, boost_dict=boost_dict)

# grid search
results = []
for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0]:
        for section_boost in [0.1, 0.5]:
            print(f'Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...')
            result = evaluate(ground_truth=ground_truth, 
                              search_function=lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                                  query=query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost))
            results.append({'question':question_boost, 'answer':answer_boost, 'section':section_boost, 'hit_rate':result['hit_rate'], 'mrr':result['mrr']})

df_results = pd.DataFrame(results)
# sort by MRR
print(f'{df_results.sort_values(by='mrr', ascending=False).head(10)}')
# the best combination weights answer twice as heavily as question, with almost no weight on section - so the data says the opposite 
# of what the intuition was - the answer text matters more for retrieval than the question text

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=2.0, section_boost=0.1...


...

Evaluating question_boost=5.0, answer_boost=4.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=4.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

    question  answer  section  hit_rate       mrr
2        1.0     2.0      0.1  0.959292  0.877316
11       2.0     4.0      0.5  0.961062  0.876991
10       2.0     4.0      0.1  0.957522  0.874897
4        1.0     4.0      0.1  0.955752  0.859735
5        1.0     4.0      0.5  0.955752  0.857670
3        1.0     2.0      0.5  0.943363  0.853982
8        2.0     2.0      0.1  0.941593  0.841239
0        1.0     1.0      0.1  0.941593  0.841003
9        2.0     2.0      0.5  0.938053  0.836018
17       5.0     4.0      0.5  0.929204  0.821917


In [ ]:
# we can use the top result and define a search function with those boost values - usually we have to look for the top k results
def text_search(query):
    boost_dict = {'question': 1.0, 'answer': 2.0, 'section': 0.1}
    return index.search(query=query, num_results=5, boost_dict=boost_dict)